In [1]:
import pandas as pd
import numpy as np
import chess
import chess.pgn
import io
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

In [16]:
def fen_to_onehot(fen: str) -> np.ndarray:
    """
    Converts a FEN string to a numpy array of shape (8,8,12) representing the board state.
    """

    board = chess.Board(fen)
    onehot = np.zeros((8,8,12), dtype=np.uint8)
    mapping = {
        chess.PAWN: 0,
        chess.KNIGHT: 1,
        chess.BISHOP: 2,
        chess.ROOK: 3,
        chess.QUEEN: 4,
        chess.KING: 5
    }

    for square in chess.SQUARES:
        piece = board.piece_at(square)
        rank = 7-chess.square_rank(square)
        file = chess.square_file(square)
        if piece:
            channel = mapping[piece.piece_type] + (6 if not piece.color else 0)
            onehot[rank, file, channel] = 1.0

    return onehot


def uci_to_index(uci: str) -> int:
    """
    Converts a UCI move to an index in the range [0, 4095].
    """

    start_file = ord(uci[0]) - ord('a')
    start_rank = int(uci[1]) - 1
    end_file = ord(uci[2]) - ord('a')
    end_rank = int(uci[3]) - 1
    start_square = 8*start_rank + start_file
    end_square = 8*end_rank + end_file
    return 64*start_square + end_square



In [17]:
df = pd.read_csv('training_data.csv')

In [18]:
df

,Board,Move,Label
0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,d2d4,0.0
1,rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR ...,d7d5,0.0
2,rnbqkbnr/ppp1pppp/8/3p4/3P4/8/PPP1PPPP/RNBQKBN...,c2c4,0.0
3,rnbqkbnr/ppp1pppp/8/3p4/2PP4/8/PP2PPPP/RNBQKBN...,c7c6,0.0
4,rnbqkbnr/pp2pppp/2p5/3p4/2PP4/8/PP2PPPP/RNBQKB...,c4d5,0.0
...,...,...,...
1212822,8/2P2k2/3Kp2p/4Pp2/3P4/6Q1/7P/2r2q2 b - - 2 37,f1a6,0.0
1212823,8/2P2k2/q2Kp2p/4Pp2/3P4/6Q1/7P/2r5 w - - 3 38,d6d7,0.0
1212824,8/2PK1k2/q3p2p/4Pp2/3P4/6Q1/7P/2r5 b - - 4 38,a6b5,0.0
1212825,8/2PK1k2/4p2p/1q2Pp2/3P4/6Q1/7P/2r5 w - - 5 39,d7d8,0.0


In [ ]:
subset = df.sample(frac=1.0)


In [ ]:
X_list = [fen_to_onehot(f) for f in fen]
y_list = [uci_to_index(u) for u in uci]


In [24]:
X_train = np.array(X_list)
y_train_indicies = np.array(y_list)
y_train = to_categorical(y_train_indicies, num_classes=4096)

print(X_train.shape)
print(y_train.shape)

(1212827, 8, 8, 12)
(1212827, 4096)


In [26]:
model = models.Sequential([
    layers.Input(shape=(8,8,12)),
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dense(4096, activation='softmax')
])



In [27]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [29]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 8, 8, 32)       │         3,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 8, 8, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     2,097,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4096)           │     2,101,248 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,220,896 (16.10 MB)

 Trainable params: 4,220,896 (16.10 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history = model.fit(X_train, y_train, batch_size=32, epochs=10, validation_split=0.1)

Epoch 1/10
34111/34111 ━━━━━━━━━━━━━━━━━━━━ 333s 10ms/step - accuracy: 0.1173 - loss: 5.1833 - val_accuracy: 0.1745 - val_loss: 3.9676
Epoch 2/10
34111/34111 ━━━━━━━━━━━━━━━━━━━━ 333s 10ms/step - accuracy: 0.1945 - loss: 3.7511 - val_accuracy: 0.1928 - val_loss: 3.7282
Epoch 3/10
34111/34111 ━━━━━━━━━━━━━━━━━━━━ 335s 10ms/step - accuracy: 0.2197 - loss: 3.4613 - val_accuracy: 0.2030 - val_loss: 3.6408
Epoch 4/10
34111/34111 ━━━━━━━━━━━━━━━━━━━━ 339s 10ms/step - accuracy: 0.2359 - loss: 3.3144 - val_accuracy: 0.2070 - val_loss: 3.6332
Epoch 5/10
34111/34111 ━━━━━━━━━━━━━━━━━━━━ 337s 10ms/step - accuracy: 0.2482 - loss: 3.2258 - val_accuracy: 0.2059 - val_loss: 3.6134
Epoch 6/10
34111/34111 ━━━━━━━━━━━━━━━━━━━━ 345s 10ms/step - accuracy: 0.2557 - loss: 3.1648 - val_accuracy: 0.2082 - val_loss: 3.6449
Epoch 7/10
34111/34111 ━━━━━━━━━━━━━━━━━━━━ 344s 10ms/step - accuracy: 0.2636 - loss: 3.1157 - val_accuracy: 0.2050 - val_loss: 3.6288
Epoch 8/10
34111/34111 ━━━━━━━━━━━━━━━━━━━━ 340s 10ms/s